In [ ]:
# --- Download semiconductor stock data over same horizon as your news (from Oct 2025) ---
# Reads your uploaded news CSV to auto-set the end date.

import pandas as pd

NEWS_PATH = "/content/labelled_semiconductor_news_vader.csv"

news = pd.read_csv(NEWS_PATH)
news["date"] = pd.to_datetime(news["date"])

start_date = "2025-10-01"
end_date = news["date"].max().strftime("%Y-%m-%d")  # last day in the news dataset

print("News date range:", news["date"].min().date(), "→", news["date"].max().date())
print("Downloading prices:", start_date, "→", end_date)

# Semiconductor tickers (edit as needed)
tickers = [
    "NVDA","AMD","INTC","TSM","AVGO","QCOM","MU","TXN","ASML",
    "AMAT","LRCX","KLAC","ADI","NXPI","MRVL","ON","MCHP","TER"
]

# Optional: add a benchmark ETF
benchmarks = ["SOXX", "SMH"]
tickers_all = tickers + benchmarks

# --- Download with yfinance ---
try:
    import yfinance as yf
except ImportError:
    raise ImportError("yfinance not installed. Run: pip install yfinance")

# yfinance end is exclusive for some endpoints; add 1 day to be safe
end_plus_one = (pd.to_datetime(end_date) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

prices = yf.download(
    tickers_all,
    start=start_date,
    end=end_plus_one,
    auto_adjust=True,   # adjusted prices (recommended)
    group_by="ticker",
    progress=False
)

# Sanity check: which tickers actually downloaded?
downloaded = sorted(set(prices.columns.get_level_values(0)))
missing = sorted(set(tickers_all) - set(downloaded))
print(f"Downloaded tickers: {len(downloaded)}")
if missing:
    print("Missing tickers (no data returned):", missing)

# --- Convert to a clean long-format table: (date, ticker, open, high, low, close, volume) ---
rows = []

for t in tickers_all:
    if t in prices.columns.get_level_values(0):
        df_t = prices[t].copy()

        # Standardise column names
        df_t.columns = [c.lower() for c in df_t.columns]

        # Reset index and standardise date column name
        df_t = df_t.reset_index()
        if "Date" in df_t.columns:
            df_t = df_t.rename(columns={"Date": "date"})
        elif "index" in df_t.columns:
            df_t = df_t.rename(columns={"index": "date"})

        df_t["ticker"] = t
        rows.append(df_t)

price_long = pd.concat(rows, ignore_index=True)

# Keep only expected columns
keep_cols = [c for c in ["date","ticker","open","high","low","close","volume"] if c in price_long.columns]
price_long = price_long[keep_cols].sort_values(["ticker","date"])

# --- Save outputs ---
out_long = "/content/semiconductor_prices_oct2025_long.csv"
price_long.to_csv(out_long, index=False)

print("Saved:", out_long)
print(price_long.head())

News date range: 2025-11-01 → 2025-12-31
Downloaded tickers: 20
Saved: /content/semiconductor_prices_oct2025_long.csv
          date ticker        open        high         low       close   volume
768 2025-10-01    ADI  242.852283  244.725656  235.388638  238.437866  4916300
769 2025-10-02    ADI  241.277830  243.111351  239.593776  240.819443  2774100
770 2025-10-03    ADI  241.198107  245.333495  240.739720  241.138321  2118500
771 2025-10-06    ADI  244.147691  244.147691  238.079128  241.646530  4021200
772 2025-10-07    ADI  242.533394  242.533394  232.209855  232.927322  3397800


In [ ]:
import numpy as np
import pandas as pd

price_long = pd.read_csv("/content/semiconductor_prices_oct2025_long.csv")
price_long["date"] = pd.to_datetime(price_long["date"])

# Sort properly
price_long = price_long.sort_values(["ticker", "date"]).reset_index(drop=True)

# Compute forward returns based on close
def add_forward_returns(df, horizons=(1,3,5)):
    df = df.copy()
    for h in horizons:
        # forward return = (close[t+h] / close[t]) - 1
        df[f"ret_fwd_{h}d"] = df["close"].shift(-h) / df["close"] - 1
    return df

price_with_rets = (
    price_long.groupby("ticker", group_keys=False)
    .apply(add_forward_returns, horizons=(1,3,5))
)

out_path = "/content/semiconductor_prices_oct2025_with_returns.csv"
price_with_rets.to_csv(out_path, index=False)

print("Saved:", out_path)
print(price_with_rets.head(10))

Saved: /content/semiconductor_prices_oct2025_with_returns.csv
        date ticker        open        high         low       close   volume  \
0 2025-10-01    ADI  242.852283  244.725656  235.388638  238.437866  4916300   
1 2025-10-02    ADI  241.277830  243.111351  239.593776  240.819443  2774100   
2 2025-10-03    ADI  241.198107  245.333495  240.739720  241.138321  2118500   
3 2025-10-06    ADI  244.147691  244.147691  238.079128  241.646530  4021200   
4 2025-10-07    ADI  242.533394  242.533394  232.209855  232.927322  3397800   
5 2025-10-08    ADI  232.977151  238.517581  232.977151  237.092606  4017900   
6 2025-10-09    ADI  236.006437  237.550985  234.272571  237.042786  3069200   
7 2025-10-10    ADI  236.903279  238.198691  223.919139  224.526993  4425100   
8 2025-10-13    ADI  227.516443  234.990047  226.161229  233.844086  4609900   
9 2025-10-14    ADI  230.336461  237.939612  228.323581  234.571503  3431200   

   ret_fwd_1d  ret_fwd_3d  ret_fwd_5d  
0    0.009988    

/tmp/ipython-input-1440445761.py:20: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(add_forward_returns, horizons=(1,3,5))


In [ ]:
import pandas as pd
import numpy as np

NEWS_PATH = "/content/labelled_semiconductor_news_vader.csv"

news = pd.read_csv(NEWS_PATH)

# Use cleaned text
TEXT_COL = "_vader_text"
LABEL_COL = "vader_label_num"

news = news[[TEXT_COL, LABEL_COL]].dropna()

# Binary label: positive vs non-positive
news["label"] = (news[LABEL_COL] == 1).astype(int)

print(news["label"].value_counts())
news.head()

label
0    157
1    135
Name: count, dtype: int64


,_vader_text,vader_label_num,label
0,ai chipmaker nvidia hits record $5 trillion ma...,0,0
1,microsoft ceo doesnt want to buy nvidia ai gpu...,1,1
2,nvidia ( nvda ) ceo is confident chinese milit...,1,1
3,"nvidia news today : korea gpu build - out , sa...",0,0
4,1 super semiconductor stock to buy hand over f...,1,1


In [ ]:
news["date"] = pd.to_datetime(pd.read_csv(NEWS_PATH)["date"])

train = news[news["date"] < "2025-12-01"]
val   = news[(news["date"] >= "2025-12-01") & (news["date"] < "2025-12-15")]
test  = news[news["date"] >= "2025-12-15"]

print(len(train), len(val), len(test))

123 71 98


In [ ]:
from collections import Counter

MAX_VOCAB = 20000
MAX_LEN = 40

def tokenize(text):
    return text.split()

counter = Counter()

for t in train[TEXT_COL]:
    counter.update(tokenize(t))

vocab = {"<PAD>": 0, "<UNK>": 1}
for word, _ in counter.most_common(MAX_VOCAB - 2):
    vocab[word] = len(vocab)

def encode(text):
    tokens = tokenize(text)
    ids = [vocab.get(tok, 1) for tok in tokens][:MAX_LEN]
    return ids + [0] * (MAX_LEN - len(ids))

X_train = np.array([encode(t) for t in train[TEXT_COL]])
X_val   = np.array([encode(t) for t in val[TEXT_COL]])
X_test  = np.array([encode(t) for t in test[TEXT_COL]])

y_train = train["label"].values
y_val   = val["label"].values
y_test  = test["label"].values

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class NewsDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

BATCH_SIZE = 32

train_loader = DataLoader(NewsDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(NewsDataset(X_val, y_val), batch_size=BATCH_SIZE)
test_loader  = DataLoader(NewsDataset(X_test, y_test), batch_size=BATCH_SIZE)

In [ ]:
import torch.nn as nn

class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        x = self.embedding(x)
        _, (h, _) = self.lstm(x)
        h = torch.cat((h[-2], h[-1]), dim=1)
        h = self.dropout(h)
        return self.fc(h).squeeze(1)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BiLSTM(len(vocab)).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def train_epoch(loader):
    model.train()
    total_loss = 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def eval_epoch(loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)
            logits = model(X)
            preds.extend(torch.sigmoid(logits).cpu().numpy())
            labels.extend(y.numpy())
    return np.array(preds), np.array(labels)

EPOCHS = 5

for epoch in range(EPOCHS):
    train_loss = train_epoch(train_loader)
    val_preds, val_labels = eval_epoch(val_loader)
    val_acc = ((val_preds > 0.5) == val_labels).mean()
    print(f"Epoch {epoch+1} | Train Loss {train_loss:.4f} | Val Acc {val_acc:.4f}")

Epoch 1 | Train Loss 0.6995 | Val Acc 0.5211
Epoch 2 | Train Loss 0.6796 | Val Acc 0.5775
Epoch 3 | Train Loss 0.6626 | Val Acc 0.5493
Epoch 4 | Train Loss 0.6461 | Val Acc 0.5775
Epoch 5 | Train Loss 0.6247 | Val Acc 0.5634


In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

test_preds, test_labels = eval_epoch(test_loader)

print(classification_report(test_labels, test_preds > 0.5))
print("ROC-AUC:", roc_auc_score(test_labels, test_preds))

              precision    recall  f1-score   support

         0.0       0.50      0.96      0.66        50
         1.0       0.00      0.00      0.00        48

    accuracy                           0.49        98
   macro avg       0.25      0.48      0.33        98
weighted avg       0.26      0.49      0.34        98

ROC-AUC: 0.5529166666666667


In [ ]:
import os
import torch

os.makedirs("Models", exist_ok=True)

torch.save(model.state_dict(), "Models/bilstm_sentiment.pt")

test_results = test.copy()
test_results["pred_prob"] = test_preds
test_results["pred_label"] = (test_preds > 0.5).astype(int)

test_results.to_csv("/content/bilstm_test_predictions.csv", index=False)

print("Saved model and predictions.")

Saved model and predictions.


In [ ]:
# Cell — BiLSTM full param grid search + save best model + save all results
# Assumes you already have: X_train, y_train, X_val, y_val, X_test, y_test, vocab, MAX_LEN
# And you already imported torch, nn, DataLoader, Dataset earlier.

import os, json, random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, roc_auc_score, precision_recall_fscore_support

os.makedirs("Models", exist_ok=True)
os.makedirs("Data", exist_ok=True)

# -----------------------
# Reproducibility
# -----------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------
# Dataset / Loaders helper
# -----------------------
class NewsDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def make_loaders(batch_size):
    train_loader = DataLoader(NewsDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(NewsDataset(X_val, y_val), batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(NewsDataset(X_test, y_test), batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader

# -----------------------
# Model
# -----------------------
class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_layers=1, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        x = self.embedding(x)
        _, (h, _) = self.lstm(x)                 # h: (num_layers*2, B, hidden_dim)
        h = torch.cat((h[-2], h[-1]), dim=1)     # last layer forward+backward
        h = self.dropout(h)
        return self.fc(h).squeeze(1)             # logits

# -----------------------
# Train/Eval helpers
# -----------------------
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total = 0.0
    for Xb, yb in loader:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(Xb)
        loss = criterion(logits, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    return total / max(len(loader), 1)

@torch.no_grad()
def predict_probs(model, loader):
    model.eval()
    probs, labels = [], []
    for Xb, yb in loader:
        Xb = Xb.to(device)
        logits = model(Xb)
        p = torch.sigmoid(logits).cpu().numpy()
        probs.append(p)
        labels.append(yb.numpy())
    return np.concatenate(probs), np.concatenate(labels)

def best_threshold_by_f1(y_true, y_prob, thresholds=np.linspace(0.1, 0.9, 33)):
    best_t, best_f1 = 0.5, -1
    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return float(best_t), float(best_f1)

def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    auc = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else float("nan")
    acc = (y_pred == y_true).mean()
    return {
        "acc": float(acc),
        "precision": float(p),
        "recall": float(r),
        "f1": float(f1),
        "roc_auc": float(auc),
        "threshold": float(threshold),
    }

# -----------------------
# Full param grid
# -----------------------
param_grid = {
    "embed_dim":   [32, 64, 128, 256],
    "hidden_dim":  [32, 64, 128, 256],
    "num_layers":  [1, 2, 3],
    "dropout":     [0.1, 0.2, 0.3, 0.4, 0.5],
    "lr":          [1e-3, 5e-4, 3e-4],
    "batch_size":  [16, 32],
    "use_pos_weight": [True, False],
    "epochs":      [50],              # keep fixed; use early stopping below
    "patience":    [2],              # early stopping on val ROC-AUC (or F1)
}

# Early stopping objective: choose one
EARLY_STOP_ON = "roc_auc"  # or "f1"

def iter_grid(grid):
    keys = list(grid.keys())
    def rec(i, cur):
        if i == len(keys):
            yield cur
            return
        k = keys[i]
        for v in grid[k]:
            nxt = dict(cur); nxt[k] = v
            yield from rec(i+1, nxt)
    yield from rec(0, {})

# Precompute pos_weight from TRAIN labels (for BCEWithLogitsLoss)
pos = y_train.sum()
neg = len(y_train) - pos
# pos_weight should be (neg/pos) for BCEWithLogitsLoss
pos_weight_tensor = torch.tensor([neg / max(pos, 1)], device=device, dtype=torch.float)

results = []
best = {
    "score": -1e9,
    "cfg": None,
    "state_dict": None,
    "val_metrics": None,
}

total_configs = sum(1 for _ in iter_grid(param_grid))
print(f"Grid configs: {total_configs}")

config_id = 0
for cfg in iter_grid(param_grid):
    config_id += 1
    print(f"\n[{config_id}/{total_configs}] cfg={cfg}")

    train_loader, val_loader, test_loader = make_loaders(cfg["batch_size"])

    model = BiLSTM(
        vocab_size=len(vocab),
        embed_dim=cfg["embed_dim"],
        hidden_dim=cfg["hidden_dim"],
        num_layers=cfg["num_layers"],
        dropout=cfg["dropout"],
    ).to(device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor) if cfg["use_pos_weight"] else nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"])

    best_epoch_score = -1e9
    best_epoch_state = None
    bad_epochs = 0

    for epoch in range(1, cfg["epochs"] + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)

        val_prob, val_true = predict_probs(model, val_loader)
        # Pick threshold on VAL to maximize F1 (helps your recall issue)
        t_star, _ = best_threshold_by_f1(val_true, val_prob)
        val_metrics = compute_metrics(val_true, val_prob, threshold=t_star)

        score = val_metrics[EARLY_STOP_ON]
        print(f"  epoch {epoch} | train_loss={train_loss:.4f} | val_auc={val_metrics['roc_auc']:.4f} "
              f"| val_f1={val_metrics['f1']:.4f} | t*={t_star:.2f}")

        if score > best_epoch_score:
            best_epoch_score = score
            best_epoch_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= cfg["patience"]:
                break

    # Evaluate best epoch state on VAL (for logging) and TEST (for reporting later)
    model.load_state_dict(best_epoch_state)

    val_prob, val_true = predict_probs(model, val_loader)
    t_star, _ = best_threshold_by_f1(val_true, val_prob)
    val_metrics = compute_metrics(val_true, val_prob, threshold=t_star)

    test_prob, test_true = predict_probs(model, test_loader)
    # IMPORTANT: use the VAL-derived threshold on TEST (no peeking)
    test_metrics = compute_metrics(test_true, test_prob, threshold=t_star)

    row = {
        **cfg,
        "early_stop_on": EARLY_STOP_ON,
        "val_acc": val_metrics["acc"],
        "val_precision": val_metrics["precision"],
        "val_recall": val_metrics["recall"],
        "val_f1": val_metrics["f1"],
        "val_roc_auc": val_metrics["roc_auc"],
        "val_threshold": val_metrics["threshold"],
        "test_acc": test_metrics["acc"],
        "test_precision": test_metrics["precision"],
        "test_recall": test_metrics["recall"],
        "test_f1": test_metrics["f1"],
        "test_roc_auc": test_metrics["roc_auc"],
    }
    results.append(row)

    # Track global best (by val metric)
    if val_metrics[EARLY_STOP_ON] > best["score"]:
        best["score"] = val_metrics[EARLY_STOP_ON]
        best["cfg"] = cfg
        best["state_dict"] = best_epoch_state
        best["val_metrics"] = val_metrics

# -----------------------
# Save grid results + best model
# -----------------------
results_df = pd.DataFrame(results).sort_values(
    by=f"val_{EARLY_STOP_ON}", ascending=False
).reset_index(drop=True)

results_csv = "Data/bilstm_grid_results.csv"
results_df.to_csv(results_csv, index=False)
print("\nSaved:", results_csv)
print("Top 5 configs:\n", results_df.head(5)[
    ["embed_dim","hidden_dim","num_layers","dropout","lr","batch_size","use_pos_weight",
     f"val_{EARLY_STOP_ON}","val_f1","val_roc_auc","test_f1","test_roc_auc"]
])

best_path = "Models/bilstm_best.pt"
torch.save(best["state_dict"], best_path)
print("\nBest model saved:", best_path)
print("Best cfg:", best["cfg"])
print("Best val metrics:", best["val_metrics"])

# Also save best config JSON for your report/repro
with open("/content/bilstm_best_config.json", "w") as f:
    json.dump({"cfg": best["cfg"], "val_metrics": best["val_metrics"], "early_stop_on": EARLY_STOP_ON}, f, indent=2)

print("Saved: Models/bilstm_best_config.json")

Streaming output truncated to the last 5000 lines.
  epoch 1 | train_loss=0.7521 | val_auc=0.4202 | val_f1=0.6078 | t*=0.10
  epoch 2 | train_loss=0.7538 | val_auc=0.4363 | val_f1=0.6078 | t*=0.10
  epoch 3 | train_loss=0.7501 | val_auc=0.4452 | val_f1=0.6078 | t*=0.10
  epoch 4 | train_loss=0.7463 | val_auc=0.4573 | val_f1=0.6078 | t*=0.10
  epoch 5 | train_loss=0.7408 | val_auc=0.4718 | val_f1=0.6078 | t*=0.10
  epoch 6 | train_loss=0.7256 | val_auc=0.4742 | val_f1=0.6078 | t*=0.10
  epoch 7 | train_loss=0.7052 | val_auc=0.4831 | val_f1=0.6078 | t*=0.10
  epoch 8 | train_loss=0.6686 | val_auc=0.4944 | val_f1=0.6078 | t*=0.10
  epoch 9 | train_loss=0.6049 | val_auc=0.5113 | val_f1=0.6078 | t*=0.10
  epoch 10 | train_loss=0.5194 | val_auc=0.5097 | val_f1=0.6078 | t*=0.10
  epoch 11 | train_loss=0.4017 | val_auc=0.5129 | val_f1=0.6078 | t*=0.10
  epoch 12 | train_loss=0.2992 | val_auc=0.5129 | val_f1=0.6078 | t*=0.10
  epoch 13 | train_loss=0.2268 | val_auc=0.4968 | val_f1=0.6078 | t*=0